In [ ]:
import sys
sys.path.append("..")   # add main_folder to path

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from typing import Tuple, Dict, Iterable, Optional
from pathlib import Path
import tomllib
from pathlib import Path
import pyarrow as pa
import pyarrow.dataset as pds
from pyarrow import compute as pc
import seaborn as sns
import matplotlib.pyplot as plt

from src.genesis.genesis_utils import read_ibtracs, preprocess_ibtracs
from src.damages.damage_functions import emanuel_2011

#Helper 
def skip_row_func(row_number):
    if (
        row_number == 1
    ):  # skip second row since it contain units, to read the types properly
        return True
    return False

catherina_fit_path = Path(
    "../../data/input/fit/Catherina_fit.db"
)

method = "RMSF"
config_path =  "../config.toml"

with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
METRIC_CRS = "EPSG:3857"   # metric for buffering (meters)
DEFAULT_BUFFER_KM = 50.0     # search radius for spatial pre-filter (change as needed)


In [ ]:
def _prepare_assets(assets: pd.DataFrame) -> gpd.GeoDataFrame:
    df = assets.copy()
    df["start_time"] = pd.to_datetime(df["start_time"], utc=True, errors="coerce")
    df["end_time"]   = pd.to_datetime(df["end_time"],   utc=True, errors="coerce")
    df = df[df["start_time"].notna() & df["end_time"].notna() & (df["end_time"] >= df["start_time"])]
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )
    return gdf

def prepare_tracks_from_pandas(tracks: pd.DataFrame, source = "IBTrACKS") -> gpd.GeoDataFrame:
    df = tracks.copy()
    if source == "IBTrACKS":
        
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
        df = df[df["datetime"].notna()]
        if "YEAR" not in df.columns:
            df["YEAR"] = df["datetime"].dt.year
        else:
            yr = df["datetime"].dt.year
            df.loc[df["YEAR"] != yr, "YEAR"] = yr
        keep = ["SID", "datetime", "YEAR", "wind", "LAT", "LON"]
        df = df[keep].sort_values(["SID", "datetime"])

        df = df.rename(columns={'LAT':'lat',
                                'LON':'lon'})
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["lon"], df["lat"]),
            crs="EPSG:4326"
        )
    elif source=="CATHERINA":
        if "SID" not in df.columns or "seed" not in df.columns or "step" not in df.columns:
            if isinstance(df.index, pd.MultiIndex):
                df = df.reset_index()
            else:
                # If only one is missing, still reset to be safe
                df = df.reset_index()

        # Choose wind column
        wind_col = "final_wind_speed" if "final_wind_speed" in df.columns else "wind_speed"
        keep = ["SID", "seed", "datetime", wind_col, "lat_left", "lon_left"]
        missing = [c for c in keep if c not in df.columns]
        if missing:
            raise ValueError(f"Missing expected columns in tracks batch: {missing}")

        df = df[keep].copy()
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
        df = df[df["datetime"].notna()]
        df = df.rename(columns={wind_col: "wind",
                                'lat_left':'lat',
                                'lon_left':'lon'})  # standardize

        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["lon"], df["lat"]),
            crs="EPSG:4326"
        )

    return gdf


In [ ]:
def exposures_vectorized_one_batch(
    assets_gdf: gpd.GeoDataFrame,
    tracks_gdf: gpd.GeoDataFrame,
    source : str,
    buffer_km: float = DEFAULT_BUFFER_KM,
) -> pd.DataFrame:
    """
    Returns long tidy table for this batch:
      columns: ['asset_id','SID','max_wind']
    """
    if assets_gdf.empty or tracks_gdf.empty:
        return pd.DataFrame(columns=["asset_id", "SID", "max_wind"])

    # 1) buffer assets
    assets_m = assets_gdf.to_crs(METRIC_CRS)
    asset_buffers = gpd.GeoDataFrame(
        assets_m[["source_id"]].copy(),
        geometry=assets_m.geometry.buffer(buffer_km * 1000.0),
        crs=assets_m.crs,
    )

    asset_points_m = assets_m[["source_id","subsector","geometry"]].rename(
        columns={"source_id":"asset_id", "geometry":"asset_point_geom"}
    )

    # 2) spatial join: track points within buffers
    tracks_m = tracks_gdf.to_crs(METRIC_CRS)
    join = gpd.sjoin(
        tracks_m[["SID", "datetime", "wind", "geometry"]],
        asset_buffers[["source_id", "geometry"]],
        predicate="within",
        how="inner",
    ).rename(columns={"source_id": "asset_id"}).drop(columns=["index_right"])


    if join.empty:
        return pd.DataFrame(columns=["asset_id", "SID", "max_wind"])

    # --- 4) Attach asset metadata (subsector + asset *point* geometry for distances)
    df = join.merge(asset_points_m, on="asset_id", how="left")


    
    #Set distance
    df = df.set_geometry("geometry")  # ensure 'geometry' column is the active one
    dist_m = df.geometry.distance(df["asset_point_geom"])
    df = df.assign(dist_m=dist_m)

    if source=='IBTrACKS':
        group_keys = ["asset_id","SID","subsector"]
        idx_min = df.groupby(group_keys, sort=False)["dist_m"].idxmin()
        df_res = df.loc[idx_min, ["asset_id","SID","subsector","wind","dist_m","datetime"]].rename(
            columns={"wind":"selected_wind", "dist_m":"nearest_dist_m", "datetime":"nearest_time"}
        ).reset_index(drop=True)
    elif source=='Simulation':
        #4) aggregate max wind per (asset_id, SID)
        df_res = (
            df.groupby(["asset_id", "SID", "subsector"], sort=False)["wind"]
            .max()
            .reset_index(name="selected_wind")
        )
    
    df_res["damages"] = emanuel_2011(v=df_res["selected_wind"])
    return df_res

def exposures_vectorized_future_periods(
    assets_gdf: gpd.GeoDataFrame,
    tracks_gdf: gpd.GeoDataFrame,
    buffer_km: float = DEFAULT_BUFFER_KM,
) -> pd.DataFrame:
    """
    Return long table: ['asset_id','SID','period','max_wind'] for this future batch.
    Periods: '2025–2050','2050–2075','2075–2100'
    """
    if assets_gdf.empty or tracks_gdf.empty:
        return pd.DataFrame(columns=["asset_id","SID","period","max_wind"])

    # Buffers
    assets_m = assets_gdf.to_crs(METRIC_CRS)
    buffers = gpd.GeoDataFrame(
        assets_m[["source_id"]].copy(),
        geometry=assets_m.geometry.buffer(buffer_km*1000.0),
        crs=assets_m.crs,
    )

    # Spatial join
    tracks_m = tracks_gdf.to_crs(METRIC_CRS)
    join = gpd.sjoin(
        tracks_m[["SID","datetime","wind","geometry"]],
        buffers[["source_id","geometry"]],
        predicate="within",
        how="inner",
    ).rename(columns={"source_id":"asset_id"})
    if join.empty:
        return pd.DataFrame(columns=["asset_id","SID","period","max_wind"])

    # Temporal mask
    windows = assets_gdf[["source_id","start_time","end_time", "subsector"]].rename(columns={"source_id":"asset_id"})
    df = join.merge(windows, on="asset_id", how="left")


    # Period binning by year
    y = df["datetime"].dt.year
    bins = [2025, 2050, 2075, 2100]  # edges (left-open/right-closed handling below)
    labels = ["2025–2050","2050–2075","2075–2100"]
    # Include edge years properly: make right=True and include_lowest=True
    df["period"] = pd.cut(y, bins=bins, labels=labels, right=True, include_lowest=True)
    df = df.dropna(subset=["period"])
    if df.empty:
        return pd.DataFrame(columns=["asset_id","SID","period","max_wind"])

    out = (
        df.groupby(["asset_id","SID","subsector","period"], sort=False)["wind"]
          .max()
          .reset_index(name="max_wind")
    )
    return out

In [ ]:
def build_historical_long_by_seed(
    tracks_root: str | Path,
    assets_df: pd.DataFrame,
    seeds :list,
    buffer_km: float = DEFAULT_BUFFER_KM
) -> pd.DataFrame:
    """
    Returns long tidy historical exposures:
      columns: ['asset_id','SID','seed','max_wind']
    Assumes Hive partitioning by 'seed' (folders like seed=123/).
    """
    assets_gdf = _prepare_assets(assets_df)
    if assets_gdf.empty:
        return pd.DataFrame(columns=["asset_id","SID","seed","max_wind"])

    dataset = pds.dataset(tracks_root, format="parquet", partitioning="hive")

    all_batches = []

    # Minimal columns we need from the dataset
    needed_cols = ["SID", "seed", "step", "datetime", "lat_left", "lon_left",
                   "final_wind_speed", "wind_speed", "SST"]

    for seed in seeds:
        # Build a type-safe filter
        filt = pds.field("seed").cast(pa.int64()) == int(seed)

        # Some columns may not exist in every fragment; Arrow will select existing ones
        scanner = pds.Scanner.from_dataset(
            dataset,
            filter=filt,
            columns=needed_cols
        )
        table = scanner.to_table()
        if table.num_rows == 0:
            continue

        df_seed = table.to_pandas()

        tracks_historical = df_seed.sort_values(["SID", "datetime"], kind="mergesort").copy().reset_index()

        #Condition on max wind
        max_wind_df = tracks_historical.groupby("SID")["final_wind_speed"].max()
        # filtering out storms with wind speed inferior to threshold
        tracks_to_keep = max_wind_df[(max_wind_df >= 32.92)&(max_wind_df <= 100)].index
        tracks_historical_filtered = tracks_historical.loc[tracks_historical["SID"].isin(tracks_to_keep)]

        # Prepare per-step GeoDataFrame
        tracks_gdf = prepare_tracks_from_pandas(tracks_historical_filtered.reset_index(),
                                                 source="CATHERINA")

        # Compute exposures for this seed
        exp_seed = exposures_vectorized_one_batch(
            assets_gdf=assets_gdf,
            tracks_gdf=tracks_gdf,
            source='Simulation',
            buffer_km=buffer_km,
        )

        if exp_seed.empty:
            continue

        exp_seed["seed"] = int(seed)
        all_batches.append(exp_seed)

    if not all_batches:
        return pd.DataFrame(columns=["asset_id","SID","seed","max_wind"])

    historical_long = pd.concat(all_batches, ignore_index=True)
    return historical_long

def build_future_long_by_seed_with_periods(
    tracks_root_future: str | Path,
    assets_df: pd.DataFrame,
    seeds :list,
    buffer_km: float = DEFAULT_BUFFER_KM
) -> pd.DataFrame:
    """
    Return long tidy future exposures with periods:
      columns: ['asset_id','SID','seed','period','max_wind']
    Periods: 2025–2050, 2050–2075, 2075–2100
    """
    assets_gdf = _prepare_assets(assets_df)
    if assets_gdf.empty:
        return pd.DataFrame(columns=["asset_id","SID","seed","period","max_wind"])

    dataset = pds.dataset(tracks_root_future, format="parquet", partitioning="hive")

    cols = ["SID","seed","step","datetime","lat_left","lon_left","final_wind_speed","wind_speed"]
    all_ = []

    for sd in seeds:
        filt = pds.field("seed").cast(pa.int64()) == int(sd)

        scanner = pds.Scanner.from_dataset(dataset, filter=filt, columns=cols)
        table = scanner.to_table()
        if table.num_rows == 0: continue

        df_seed = table.to_pandas()
        tracks_gdf = prepare_tracks_from_pandas(df_seed.reset_index(), 
                                                source="CATHERINA")
        out = exposures_vectorized_future_periods(assets_gdf, tracks_gdf, buffer_km)
        if out.empty: continue
        out["seed"] = sd
        all_.append(out)

    if not all_:
        return pd.DataFrame(columns=["asset_id","SID","seed","period","max_wind"])
    return pd.concat(all_, ignore_index=True)


In [ ]:
#Import data: assets
assets_path = Path("../../data/input/assets/ctrace.csv")
# Define filter criteria
sectors = ['fossil-fuel-operations', 'manufacturing', 'mineral-extraction', 'power']
# Read in chunks, keep only USA + required sectors + valid lat
chunks = []
for chunk in pd.read_csv(assets_path, chunksize=100_000):  # adjust chunksize as needed
    filtered = (
        chunk.loc[chunk['iso3_country'] == 'USA']
             .loc[~chunk['lat'].isna()]
             .loc[chunk['sector'].isin(sectors)]
    )
    chunks.append(filtered)
# Concatenate the filtered chunks
assets_sector = pd.concat(chunks, ignore_index=True)
# Drop duplicate source_ids
assets_sector = assets_sector.drop_duplicates(subset="source_id")

In [ ]:
#Read and preprocess ibtracks

#Read and preprocess ibtracks
main_config = config_files["main_params"]
gen_config = config_files["generation"]
data_dir = ".." / Path(main_config["input_data_dir"])
ibtracksf_folder = data_dir / gen_config["ibtracs_path"]

#Theo version
#ibtracs_theo = process_ibtracs_data(ibtracksf_folder,2014, 32.92)

#New version
ibtracs = read_ibtracs(fpath=ibtracksf_folder, signed_coords=True)
ibtracs = preprocess_ibtracs(ibtracs, 32.92)


In [ ]:
tracks_ibtracks_gdf = prepare_tracks_from_pandas(ibtracs, source="IBTrACKS")
assets_gdf = _prepare_assets(assets_sector)

ibtracks_wind_long = exposures_vectorized_one_batch(assets_gdf,tracks_ibtracks_gdf , "IBTrACKS")

exposed_assets_sector = assets_sector.loc[lambda row:row['source_id'].isin(ibtracks_wind_long['asset_id'].unique().tolist())]

In [ ]:
tracks_historical_dir = Path("../../data/input/catherina_historical/intensified_tracks_test_base_param/ACCESS-CM2/historical/")

historical_long = build_historical_long_by_seed(
    tracks_root=tracks_historical_dir,  # root directory with seed=.../ partitions
    assets_df=exposed_assets_sector,
    seeds=list(range(20))
)

In [ ]:
historical_long

In [ ]:
sns.histplot(historical_long.groupby('asset_id')['damages'].mean())

In [ ]:
import seaborn as sns
sns.histplot(ibtracks_wind_long.loc[:, 'damages'])

In [ ]:
import seaborn as sns
sns.ecdfplot(ibtracks_wind_long.loc[lambda row:row['damages']>0, 'damages'])
sns.ecdfplot(historical_long.loc[lambda row:(row['damages']>0),:].groupby('asset_id')['damages'].quantile(0.1))
sns.ecdfplot(historical_long.loc[lambda row:(row['damages']>0),:].groupby('asset_id')['damages'].quantile(0.5))
sns.ecdfplot(historical_long.loc[lambda row:(row['damages']>0),:].groupby('asset_id')['damages'].quantile(0.9))

In [ ]:
historical_long.loc[lambda row:row['damages']>0, 'damages'].mean()

In [ ]:
tracks_ssp585_dir = Path("../data/input/catherina_test4/intensified_tracks/ACCESS-CM2/ssp585/")

future_long = build_future_long_by_seed_with_periods(
    tracks_root_future=tracks_ssp585_dir,  # root directory with seed=.../ partitions
    assets_df=assets_sector,
    seeds=list(range(1)),
    buffer_km=250.0
)

In [ ]:
import math
def plot_distributions_by_subsector(
    exposures_current_long: pd.DataFrame,
    historical_long: pd.DataFrame,
    max_wind_col: str = "max_wind",
    ncols: int = 4,
    figsize_per_panel: tuple[float, float] = (4.0, 3.2),
    sharex: bool = False,
    sharey: bool = False,
):
    """
    One figure with subplots arranged in 4 columns (default) and as many rows as needed.
    For each subsector:
      - plot historical distributions (all seeds, colored by seed)
      - overlay current distribution (black)
    """

    # Guard: keep only finite winds
    cur = exposures_current_long[np.isfinite(exposures_current_long[max_wind_col])]
    hist = historical_long[np.isfinite(historical_long[max_wind_col])]

    subsectors = np.sort(cur["subsector"].dropna().unique())
    if len(subsectors) == 0:
        raise ValueError("No subsectors found in current exposures (after dropping NaNs).")

    n = len(subsectors)
    nrows = math.ceil(n / ncols)
    fig_w = figsize_per_panel[0] * ncols
    fig_h = figsize_per_panel[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharex=sharex, sharey=sharey, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for i, s in enumerate(subsectors):
        ax = axes[i]

        # Historical: per-seed KDEs
        h_sub = hist[hist["subsector"] == s]
        if not h_sub.empty and "seed" in h_sub.columns:
            sns.kdeplot(
                data=h_sub,
                x=max_wind_col,
                hue="seed",
                alpha=0.30,
                common_norm=False,
                linewidth=1.0,
                ax=ax,
                legend=False,     # many seeds → legend gets noisy
            )

        # Current: black KDE
        c_sub = cur[cur["subsector"] == s]
        if not c_sub.empty:
            sns.kdeplot(
                data=c_sub,
                x=max_wind_col,
                color="black",
                linewidth=2.0,
                ax=ax,
                label="IBTrACKS",
            )

        ax.set_title(f"{s}", fontsize=11)
        ax.set_xlabel("Max wind speed (m/s)")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.25)

        # Optional: small legend (only shows 'Current')
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(loc="upper right", fontsize=8, frameon=False)

    # Hide any unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Max wind distributions by subsector", fontsize=14, y=1.02)
    plt.tight_layout()
    return fig

In [ ]:
def plot_box_violin_by_subsector(
    exposures_current_long: pd.DataFrame,   # ['asset_id','SID','max_wind']
    historical_long: pd.DataFrame,          # ['asset_id','SID','seed','max_wind']
    plot_kind: str = "violin"                  # "box" or "violin"
):
    """
    Compare distributions of max_wind (current vs historical) per subsector.

    Parameters
    ----------
    assets_df : DataFrame
        Must include ['source_id','subsector'] at least.
    exposures_current_long : DataFrame
        Current exposures ['asset_id','SID','max_wind'].
    historical_long : DataFrame
        Historical exposures ['asset_id','SID','seed','max_wind'].
    plot_kind : str
        'box' for boxplot, 'violin' for violin plot.
    """

    # Attach subsector

    # Build combined tidy table
    cur_plot = exposures_current_long[["subsector","max_wind"]].assign(dataset="current")
    hist_plot = historical_long[["subsector","max_wind"]].assign(dataset="historical")
    combined = pd.concat([cur_plot, hist_plot], ignore_index=True)
    combined = combined.dropna(subset=["subsector","max_wind"])

    subsectors = combined["subsector"].dropna().unique()
    subsectors = sorted(subsectors)

    for s in subsectors:
        data_s = combined[combined["subsector"] == s]
        if data_s.empty:
            continue

        plt.figure(figsize=(8,5))
        if plot_kind == "box":
            sns.boxplot(
                data=data_s,
                x="max_wind", y="dataset",
                orient="h",
                showfliers=True,  # to display outliers
                palette={"current": "lightcoral", "historical": "skyblue"}
            )
        elif plot_kind == "violin":
            sns.violinplot(
                data=data_s,
                x="max_wind", y="dataset",
                orient="h",
                cut=0,
                palette={"current": "lightcoral", "historical": "skyblue"}
            )
        else:
            raise ValueError("plot_kind must be 'box' or 'violin'")

        plt.title(f"Max wind distribution — {s}")
        plt.xlabel("Max wind speed (m/s)")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()


In [ ]:
distribution = plot_distributions_by_subsector(ibtracks_wind_long,historical_long, max_wind_col='max_wind')

In [ ]:
def _ecdf_on_grid(sample: np.ndarray, grid: np.ndarray) -> np.ndarray:
    """ECDF(sample) evaluated on a fixed grid."""
    if sample is None or len(sample) == 0:
        return np.full_like(grid, np.nan, dtype=float)
    s = np.sort(np.asarray(sample, dtype=float))
    return np.searchsorted(s, grid, side="right") / s.size

def plot_ecdf_by_subsector(
    exposures_current_long: pd.DataFrame,   # no seeds (or one scenario)
    historical_long: pd.DataFrame,          # has multiple seeds in column 'seed'
    subsector_col: str = "subsector",
    value_col: str = "max_wind",
    seed_col: str = "seed",
    ncols: int = 4,
    figsize_per_panel: tuple[float, float] = (4.0, 3.2),
    sharex: bool = True,
    sharey: bool = True,
    q_low: float = 0.10,                    # CI lower quantile across seeds
    q_high: float = 0.90,                   # CI upper quantile across seeds
    n_grid: int = 400,
    title: str = "ECDF of max wind by subsector (seed-aware CI for historical)",
):
    """
    For each subsector:
      - Historical: ECDF per seed -> plot mean ECDF (line) + quantile band [q_low, q_high]
      - Current: ECDF (black line)
    All panels in one figure with 4 columns and as many rows as needed.
    """

    # Keep only finite values
    cur = exposures_current_long[np.isfinite(exposures_current_long[value_col])].copy()
    hist = historical_long[np.isfinite(historical_long[value_col])].copy()

    # Subsector list: union (so subsectors present only in one side still get a panel)
    subsectors = np.sort(
        np.union1d(
            cur[subsector_col].dropna().unique() if subsector_col in cur else [],
            hist[subsector_col].dropna().unique() if subsector_col in hist else []
        )
    )
    if len(subsectors) == 0:
        raise ValueError("No subsectors found (after dropping NaNs).")

    n = len(subsectors)
    nrows = math.ceil(n / ncols)
    fig_w = figsize_per_panel[0] * ncols
    fig_h = figsize_per_panel[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h),
                             sharex=sharex, sharey=sharey, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    # Precompute global x-limits for consistency (optional; can be per-panel)
    global_vals = []
    if not cur.empty: global_vals.append(cur[value_col].to_numpy())
    if not hist.empty: global_vals.append(hist[value_col].to_numpy())
    if global_vals:
        gmin = np.nanpercentile(np.concatenate(global_vals), 0.5)
        gmax = np.nanpercentile(np.concatenate(global_vals), 99.5)
        if not np.isfinite(gmin) or not np.isfinite(gmax) or gmin >= gmax:
            gmin, gmax = float(np.nanmin(np.concatenate(global_vals))), float(np.nanmax(np.concatenate(global_vals)))
    else:
        gmin, gmax = 0.0, 1.0  # fallback

    for i, s in enumerate(subsectors):
        ax = axes[i]

        # Panel-specific data
        cur_s = cur
        hist_s = hist
        #cur_s = cur[cur[subsector_col] == s] if not cur.empty else cur
        #hist_s = hist[hist[subsector_col] == s] if not hist.empty else hist

        # Build grid for ECDF (use panel-specific data but clamp within global range)
        pool = []
        if not cur_s.empty: pool.append(cur_s[value_col].to_numpy())
        if not hist_s.empty: pool.append(hist_s[value_col].to_numpy())
        if pool:
            x_min, x_max = np.nanpercentile(np.concatenate(pool), [0.5, 99.5])
            if not np.isfinite(x_min) or not np.isfinite(x_max) or x_min >= x_max:
                x_min, x_max = float(np.nanmin(np.concatenate(pool))), float(np.nanmax(np.concatenate(pool)))
            # enforce global limits for visual comparability
            x_min = max(x_min, gmin)
            x_max = min(x_max, gmax)
        else:
            x_min, x_max = gmin, gmax
        grid = np.linspace(x_min, x_max, n_grid)

        # --- Historical: seed-aware ECDF mean + band ---
        if not hist_s.empty and seed_col in hist_s.columns:
            ecdfs = []
            for _, g_seed in hist_s.groupby(seed_col, sort=False):
                sample = g_seed[value_col].to_numpy()
                ecdfs.append(_ecdf_on_grid(sample, grid))
            if len(ecdfs) > 0:
                M = np.vstack(ecdfs)
                mean_ecdf = np.nanmean(M, axis=0)
                lo = np.nanquantile(M, q_low, axis=0)
                hi = np.nanquantile(M, q_high, axis=0)
                ax.step(grid, mean_ecdf, where="post", label="Historical (mean over seeds)", linewidth=1.6, color="tab:blue")
                ax.fill_between(grid, lo, hi, step="post", alpha=0.20, color="tab:blue",
                                label=f"Historical [{int(q_low*100)}–{int(q_high*100)}%]")

        # --- Current: single ECDF ---
        if not cur_s.empty:
            ecdf_cur = _ecdf_on_grid(cur_s[value_col].to_numpy(), grid)
            ax.step(grid, ecdf_cur, where="post", label="IBTrACKS", linewidth=2.0, color="black")

        ax.set_title(f"{s}", fontsize=11)
        ax.set_xlabel("Max wind speed (m/s)")
        ax.set_ylabel("ECDF")
        ax.set_xlim(grid[0], grid[-1])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.25)

        # Tidy legend
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(loc="lower right", fontsize=8, frameon=False)

    # Hide any unused panels
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    return fig


def plot_ecdf_by_category(
    exposures_current_long: pd.DataFrame,   # no seeds (or one scenario)
    historical_long: pd.DataFrame,          # has multiple seeds in column 'seed'
    category_col: str = "sector",           # <-- use "sector" here (instead of "subsector")
    value_col: str = "max_wind",
    seed_col: str = "seed",
    ncols: int = 4,
    figsize_per_panel: tuple[float, float] = (4.0, 3.2),
    sharex: bool = True,
    sharey: bool = True,
    q_low: float = 0.10,                    # CI lower quantile across seeds
    q_high: float = 0.90,                   # CI upper quantile across seeds
    n_grid: int = 400,
    title: str = "ECDF of max wind by sector (seed-aware CI for historical)",
    label_current: str = "Current",
    label_hist_mean: str = "Historical (mean over seeds)",
    label_hist_band: str = None,            # if None -> formatted from q_low/q_high
):
    """
    For each category (e.g., sector):
      - Historical: ECDF per seed -> plot mean ECDF (line) + quantile band [q_low, q_high]
      - Current: ECDF (black line)
    All panels in one figure with `ncols` columns and as many rows as needed.

    Notes
    -----
    - `historical_long` should have a column `seed` with the seed identifier.
    - The function is robust when a category appears in only one of the two datasets.
    """

    # --- small helper so the function is self-contained
    def _ecdf_on_grid(sample: np.ndarray, grid: np.ndarray) -> np.ndarray:
        """Right-continuous ECDF evaluated on `grid`."""
        x = np.sort(sample[np.isfinite(sample)])
        if x.size == 0:
            return np.zeros_like(grid, dtype=float)
        # For ECDF F(t) = P(X ≤ t); use searchsorted with 'right'
        return np.searchsorted(x, grid, side="right") / x.size

    # Keep only finite values
    cur = exposures_current_long[np.isfinite(exposures_current_long[value_col])].copy()
    hist = historical_long[np.isfinite(historical_long[value_col])].copy()

    # Category list: union (categories present in only one side still get a panel)
    cats_cur  = cur[category_col].dropna().unique() if (not cur.empty and category_col in cur) else np.array([])
    cats_hist = hist[category_col].dropna().unique() if (not hist.empty and category_col in hist) else np.array([])
    categories = np.sort(np.union1d(cats_cur, cats_hist))
    if len(categories) == 0:
        raise ValueError(f"No categories found in column '{category_col}' (after dropping NaNs).")

    n = len(categories)
    nrows = math.ceil(n / ncols)
    fig_w = figsize_per_panel[0] * ncols
    fig_h = figsize_per_panel[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h),
                             sharex=sharex, sharey=sharey, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    # Precompute global x-limits for consistency (optional)
    global_vals = []
    if not cur.empty:  global_vals.append(cur[value_col].to_numpy())
    if not hist.empty: global_vals.append(hist[value_col].to_numpy())
    if global_vals:
        gmin = np.nanpercentile(np.concatenate(global_vals), 0.5)
        gmax = np.nanpercentile(np.concatenate(global_vals), 99.5)
        if (not np.isfinite(gmin)) or (not np.isfinite(gmax)) or (gmin >= gmax):
            gmin = float(np.nanmin(np.concatenate(global_vals)))
            gmax = float(np.nanmax(np.concatenate(global_vals)))
    else:
        gmin, gmax = 0.0, 1.0  # fallback

    # Default label for band
    if label_hist_band is None:
        label_hist_band = f"Historical [{int(q_low*100)}–{int(q_high*100)}%]"

    for i, c in enumerate(categories):
        ax = axes[i]

        # Panel-specific data (FIX: actually filter by category)
        cur_c  = cur[cur[category_col] == c] if (not cur.empty and category_col in cur.columns) else cur
        hist_c = hist[hist[category_col] == c] if (not hist.empty and category_col in hist.columns) else hist

        # Build grid for ECDF (clamped to global range for visual comparability)
        pool = []
        if not cur_c.empty:  pool.append(cur_c[value_col].to_numpy())
        if not hist_c.empty: pool.append(hist_c[value_col].to_numpy())

        if pool:
            x_min, x_max = np.nanpercentile(np.concatenate(pool), [0.5, 99.5])
            if (not np.isfinite(x_min)) or (not np.isfinite(x_max)) or (x_min >= x_max):
                x_min = float(np.nanmin(np.concatenate(pool)))
                x_max = float(np.nanmax(np.concatenate(pool)))
            x_min = max(x_min, gmin)
            x_max = min(x_max, gmax)
        else:
            x_min, x_max = gmin, gmax

        grid = np.linspace(x_min, x_max, n_grid)

        # --- Historical: seed-aware ECDF mean + band ---
        if (not hist_c.empty) and (seed_col in hist_c.columns):
            ecdfs = []
            for _, g_seed in hist_c.groupby(seed_col, sort=False):
                sample = g_seed[value_col].to_numpy()
                ecdfs.append(_ecdf_on_grid(sample, grid))
            if len(ecdfs) > 0:
                M = np.vstack(ecdfs)
                mean_ecdf = np.nanmedian(M, axis=0)
                lo = np.nanquantile(M, q_low, axis=0)
                hi = np.nanquantile(M, q_high, axis=0)
                ax.step(grid, mean_ecdf, where="post",
                        label=label_hist_mean, linewidth=1.6, color="tab:blue")
                ax.fill_between(grid, lo, hi, step="post", alpha=0.20, color="tab:blue",
                                label=label_hist_band)

        # --- Current: single ECDF ---
        if not cur_c.empty:
            ecdf_cur = _ecdf_on_grid(cur_c[value_col].to_numpy(), grid)
            ax.step(grid, ecdf_cur, where="post", label=label_current, linewidth=2.0, color="black")

        ax.set_title(f"{c}", fontsize=11)
        ax.set_xlabel("Max wind speed (m/s)")
        ax.set_ylabel("ECDF")
        ax.set_xlim(grid[0], grid[-1])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.25)

        # Tidy legend
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(loc="lower right", fontsize=8, frameon=False)

    # Hide any unused panels
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    return fig




In [ ]:
fig = plot_ecdf_by_category(
    exposures_current_long=ibtracks_wind_long.loc[lambda row:row['damages']>0,].assign(sector="All"),
    historical_long=historical_long.loc[lambda row:row['damages']>0,].assign(sector="All"),
    category_col="sector",
    value_col="damages",
    seed_col="seed",
    ncols=4,
    q_low=0.1, 
    q_high=0.90,
    title="ECDF of max wind (all sectors combined)"
)